# JAX: Dense GEMMs with TransformerEngine

This notebook walks through replacing a plain `flax.linen.Dense`'s GEMM with TransformerEngine's quantized GEMM.

**Recipe.** We use `MXFP8BlockScaling` in this tutorial. `MXFP8BlockScaling` and `NVFP4BlockScaling` require a Blackwell-class GPU; on Hopper, swap in `DelayedScaling` or `Float8CurrentScaling`.

[← Back to the JAX integration overview](../te_jax_integration.ipynb)

## 1. Baseline: a plain Flax Dense block

We isolate the optimization to a single linear layer so it's clear what's changing. `dot_general_cls` is exposed as a constructor argument so we can swap in TE later without touching the model definition.

In [ ]:
import sys
sys.path.append("..")  # so we can import quickstart_jax_utils from docs/examples/

import warnings

import jax
import jax.numpy as jnp
from flax import linen as nn
import quickstart_jax_utils as utils

In [ ]:
class FlaxDenseBlock(nn.Module):
    """One linear layer. `dot_general_cls` lets us swap the GEMM impl."""
    features: int
    dot_general_cls: callable = lambda: None

    @nn.compact
    def __call__(self, x):
        return nn.Dense(
            features=self.features,
            use_bias=False,
            dot_general=self.dot_general_cls(),
        )(x)

batch, seq, hidden, out_features = 4, 2048, 4096, 16384
dtype = jnp.bfloat16

key = jax.random.PRNGKey(0)
k_init, k_x, k_dy = jax.random.split(key, 3)
x = jax.random.normal(k_x, (batch, seq, hidden)).astype(dtype)
dy = jax.random.normal(k_dy, (batch, seq, out_features)).astype(dtype)

baseline = FlaxDenseBlock(features=out_features)
baseline_vars = baseline.init(k_init, x)

## 2. Quantized Dense via `make_dot_general_cls`

TE exposes a helper, `te_flax.make_dot_general_cls(recipe)`, that returns a Flax module class you pass directly to `nn.Dense(..., dot_general=...)`.

With this API, TE doesn't create the `kernel` params; it only wraps the GEMM. All your initialization, sharding annotations, and optimizer state stay where they were.

In [3]:
from transformer_engine.jax import flax as te_flax
from transformer_engine.common.recipe import MXFP8BlockScaling

recipe = MXFP8BlockScaling()
te_dot_general_cls = te_flax.make_dot_general_cls(recipe)

te_model = FlaxDenseBlock(features=out_features, dot_general_cls=te_dot_general_cls)
te_vars = te_model.init(k_init, x)

print("Variable collections:", list(te_vars.keys()))
print(jax.tree_util.tree_map(lambda a: (a.shape, a.dtype), te_vars))

Variable collections: ['params']
{'params': {'Dense_0': {'kernel': ((4096, 16384), dtype('float32'))}}}


<div class="alert alert-info">
<p><b>What about DelayedScaling state?</b></p>

Most recipes are stateless — scaling factors are computed from each tensor as it flows through the GEMM, so there is nothing to persist across steps. However, if you swap in `DelayedScaling` instead, `init` will produce a second variable collection, `_overwrite_with_gradient`, holding `kernel_amax_history`, `kernel_scale`, `x_amax_history`, `x_scale`, etc. These are **not** model parameters — they are Flax variables that TE updates each step to compute per-tensor scales from a rolling amax window.

If you use `DelayedScaling`, you must thread the *entire* `var_collect` through your training loop (not just `params`) so the history persists across steps. `MXFP8BlockScaling`, `NVFP4BlockScaling`, and `Float8CurrentScaling` do not require this.
</div>

## 3. Single-GPU performance

`speedometer` runs a JIT-compiled forward+backward loop with warmup, on the same input for both models.

In [4]:
print("bf16 baseline:")
utils.speedometer(
    model_apply_fn=baseline.apply,
    variables=baseline_vars,
    input=x, output_grad=dy,
)

print(f"\nTE {type(recipe).__name__}:")
utils.speedometer(
    model_apply_fn=te_model.apply,
    variables=te_vars,
    input=x, output_grad=dy,
)

bf16 baseline:


Mean time: 4.126272201538086 ms

TE MXFP8BlockScaling:


E0511 14:02:41.217310  154511 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


Mean time: 1.689767837524414 ms


On a single GB200, that's roughly **2.5× faster** for the fwd+bwd of one large Dense — and the only code change was passing `dot_general=te_dot_general_cls()` into `nn.Dense`.

The speedup depends on shape: large GEMMs benefit most. Very small GEMMs may not benefit at all because the cast + scale overhead can dominate.

<div class="alert alert-warning">
<b>Remat / activation checkpointing.</b> If your training loop uses <code>jax.checkpoint_policies.checkpoint_dots</code> (or any policy that matches <code>jax.lax.dot_general</code>), swap it for <code>transformer_engine.jax.checkpoint_policies.checkpoint_dots_and_te_gemms</code>. Otherwise TE's quantized GEMM primitives won't be checkpointed correctly and your performance comparison will not be accurate.
</div>

## 6. Multi-GPU: DP=2 / TP=2 on a single Dense

**Prerequisite:** this section requires four GPUs.

Keeping the same `FlaxDenseBlock` from the rest of the notebook, we run it on a 2×2 mesh with **data parallelism** on one axis and **tensor parallelism** (column-parallel: shard the kernel's output dim) on the other.

Two pieces wire this up:

1. A `jax.sharding.Mesh` you build once at module scope (outside JIT).
2. TE's `MeshResource`, set globally via `global_shard_guard`, which tells TE which mesh axes are DP and TP.

In [6]:
n_devices = len(jax.devices())
print(f"Visible devices: {n_devices}")
assert n_devices >= 4, "This section requires 4 GPUs for DP=2/TP=2."

Visible devices: 4


In [7]:
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P
from jax.experimental import mesh_utils
from transformer_engine.jax.sharding import MeshResource, global_shard_guard

# 2x2 mesh: DP on one axis, TP on the other.
devices = mesh_utils.create_device_mesh((2, 2))
mesh = Mesh(devices, axis_names=("dp", "tp"))

# Tell TE which mesh axis is which. This is a *global* setting, established
# outside JIT, so TE's GEMM primitives can plan comms accordingly.
mesh_resource = MeshResource(dp_resource="dp", tp_resource="tp")

**Sharding plan:**

| Tensor | Shape | PartitionSpec |
|---|---|---|
| Kernel (column-parallel) | `(hidden, out_features)` | `P(None, "tp")` |
| Input activations | `(batch, seq, hidden)` | `P("dp", None, None)` |
| Gradient on output | `(batch, seq, out_features)` | `P("dp", None, "tp")` |

In [8]:
kernel_sharding = NamedSharding(mesh, P(None, "tp"))
input_sharding = NamedSharding(mesh, P("dp", None, None))
output_grad_sharding = NamedSharding(mesh, P("dp", None, "tp"))

def shard_kernel(variables):
    params = variables["params"]
    sharded = jax.device_put(params["Dense_0"]["kernel"], kernel_sharding)
    return {**variables, "params": {**params,
                                    "Dense_0": {**params["Dense_0"], "kernel": sharded}}}

x_mp_s = jax.device_put(x, input_sharding)
dy_mp_s = jax.device_put(dy, output_grad_sharding)
baseline_vars_s = shard_kernel(baseline_vars)
te_vars_s = shard_kernel(te_vars)

In [9]:
with jax.set_mesh(mesh), global_shard_guard(mesh_resource):
    print("bf16 DP=2/TP=2:")
    utils.speedometer(
        model_apply_fn=baseline.apply,
        variables=baseline_vars_s,
        input=x_mp_s, output_grad=dy_mp_s,
    )

    print(f"\nTE {type(recipe).__name__} DP=2/TP=2:")
    utils.speedometer(
        model_apply_fn=te_model.apply,
        variables=te_vars_s,
        input=x_mp_s, output_grad=dy_mp_s,
    )

bf16 DP=2/TP=2:


Mean time: 1.7258834838867188 ms

TE MXFP8BlockScaling DP=2/TP=2:


Mean time: 0.9692144393920898 ms


## 7. Collective GEMM (placeholder)

*Coming soon.*

**Next:** [Attention](./attention.ipynb) · [Mixture of Experts](./moe.ipynb) · [← Hub](../te_jax_integration.ipynb)